# Research paper exploration

Querying everything `save_papers` and `research_brief` have written to `company_context_documents` (`source = 'semantic_scholar'`). Useful for confirming a Slack academic-research turn actually persisted, and for exploring what the agent's `semantic_scholar.search` indexed lane sees. For Slack channel data, see [`slack.ipynb`](slack.ipynb).

Connection plumbing lives in the `centaur_db` package — `db.connect()` starts a kubectl port-forward to the in-cluster Postgres if one isn't already up, and tears it down on kernel shutdown.

Use `db.query(conn, sql, params)` for SELECTs (returns a `pandas.DataFrame`) and `db.execute(conn, sql, params)` for INSERT/UPDATE/DELETE (returns rowcount).

In [ ]:
import centaur_db as db

conn = db.connect()
q = lambda sql, params=None: db.query(conn, sql, params)
q("SELECT current_database() AS db, version() AS pg_version").iloc[0].to_dict()

## Semantic Scholar — saved papers and briefs

Queries `company_context_documents` for everything the `save_papers` and `research_brief` workflows have written (`source = 'semantic_scholar'`). Useful for confirming that a Slack academic-research turn actually persisted, and for exploring what the agent's BM25 retriever sees in the **indexed lane** of `semantic_scholar.search`.

The shape lines up with how the workflows write rows:

- `source_type = 'paper'` — one row per persisted Semantic Scholar paper. `metadata` carries the full S2 record (year, citation count, authors, externalIds) plus the traceability query.
- `source_type = 'research_brief'` — one row per generated brief (markdown in `body`). Child papers link via `parent_document_id` → brief `document_id`; use the join in the section below to list a brief's papers.

In [ ]:
q(
    "SELECT "
    "  source_type, "
    "  left(title, 80) AS title, "
    "  author_name, "
    "  (metadata->>'year')::int AS year, "
    "  (metadata->>'citationCount')::int AS citations, "
    "  length(body) AS body_len, "
    "  parent_document_id IS NOT NULL AS has_parent, "
    "  source_updated_at "
    "FROM company_context_documents "
    "WHERE source = 'semantic_scholar' "
    "ORDER BY source_updated_at DESC "
    "LIMIT 50"
)

### Briefs and their child papers

Each `research_brief` row groups N child paper rows via `parent_document_id`. This view shows every brief on file, the query that triggered it, and how many papers it bundled — a quick way to see what the agent has been researching.

In [ ]:
q(
    "SELECT "
    "  b.metadata->>'query' AS query, "
    "  left(b.title, 60) AS brief_title, "
    "  count(p.document_id) AS paper_count, "
    "  b.source_updated_at "
    "FROM company_context_documents b "
    "LEFT JOIN company_context_documents p "
    "  ON p.parent_document_id = b.document_id "
    "  AND p.source = 'semantic_scholar' "
    "  AND p.source_type = 'paper' "
    "WHERE b.source = 'semantic_scholar' "
    "  AND b.source_type = 'research_brief' "
    "GROUP BY b.document_id, b.title, b.metadata, b.source_updated_at "
    "ORDER BY b.source_updated_at DESC"
)

### Drill into one brief's papers

Child papers link to their brief via `parent_document_id` (authoritative). Both brief and papers also store the original query in `metadata->>'query'` for filtering. Set `BRIEF_QUERY` to a substring of the query you care about, then run the join below.

In [ ]:
BRIEF_QUERY = "active inference"

q(
    """
    SELECT
      p.title,
      p.author_name,
      (p.metadata->>'year')::int AS year,
      (p.metadata->>'citationCount')::int AS citations,
      p.metadata->>'paperId' AS paper_id,
      p.parent_document_id IS NOT NULL AS has_parent,
      p.source_updated_at
    FROM company_context_documents b
    JOIN company_context_documents p
      ON p.parent_document_id = b.document_id
    WHERE b.source = 'semantic_scholar'
      AND b.source_type = 'research_brief'
      AND b.metadata->>'query' ILIKE %s
      AND p.source = 'semantic_scholar'
      AND p.source_type = 'paper'
    ORDER BY (p.metadata->>'citationCount')::int DESC NULLS LAST
    """,
    (f"%{BRIEF_QUERY}%",),
)

Same papers filtered by `metadata->>'query'` on the paper rows alone. Useful when comparing against orphan papers saved before brief linking (`has_parent = false`).

In [ ]:
q(
    """
    SELECT
      title,
      author_name,
      (metadata->>'year')::int AS year,
      (metadata->>'citationCount')::int AS citations,
      parent_document_id IS NOT NULL AS has_parent,
      source_updated_at
    FROM company_context_documents
    WHERE source = 'semantic_scholar'
      AND source_type = 'paper'
      AND metadata->>'query' ILIKE %s
    ORDER BY source_updated_at DESC
    """,
    (f"%{BRIEF_QUERY}%",),
)

### Brief markdown body

The rendered lit-review lives in the brief row's `body` column (same markdown the agent can post to Slack).

In [ ]:
brief = q(
    """
    SELECT title, body, metadata->>'query' AS query, source_updated_at
    FROM company_context_documents
    WHERE source = 'semantic_scholar'
      AND source_type = 'research_brief'
      AND metadata->>'query' ILIKE %s
    ORDER BY source_updated_at DESC
    LIMIT 1
    """,
    (f"%{BRIEF_QUERY}%",),
)
brief.iloc[0]["body"] if len(brief) else "No brief matched BRIEF_QUERY"

### BM25 over saved Semantic Scholar rows

This is exactly what the agent's `semantic_scholar.search` indexed lane runs under the hood — same `body @@@` operator, same `paradedb.score` ranking. Run it locally to see what BM25 hits a query produces against the cache, before the agent ever calls the live S2 API.

In [ ]:
q(
    "SELECT "
    "  source_type, "
    "  left(title, 60) AS title, "
    "  paradedb.score(document_id) AS score, "
    "  (metadata->>'year')::int AS year, "
    "  left(body, 240) AS preview "
    "FROM company_context_documents "
    "WHERE source = 'semantic_scholar' "
    "  AND body @@@ %s "
    "ORDER BY score DESC LIMIT 10",
    ("attention mechanism transformer",),
)